In [ ]:
"""
Courant I(Φ) et conductance G(Φ) — SIAM (Modèle d'Anderson)
HierarchicalEOM.jl — deux bains L/R, 4 canaux spin, HEOM exact

═══════════════════════════════════════════════════════════════════
AUDIT DU CODE ORIGINAL — CORRECTIONS APPLIQUÉES
═══════════════════════════════════════════════════════════════════

1. Γ était une liste passée à Fermion_Lorentz_Pade → crash (scalaire requis)
   → boucle `for Gamma in Gamma_list` avec reconstruction des bains à chaque Γ

2. HEOMsolve1(...).ados → fonction inexistante
   → steadystate(M_even) à chaque (Γ, Φ) — API officielle

3. Un seul bain μ=0 pour les deux canaux : physiquement incorrect
   → 4 bains : bath_up_L (μ=+Φ/2), bath_up_R (μ=-Φ/2),
               bath_dn_L (μ=+Φ/2), bath_dn_R (μ=-Φ/2)
   → bathIdx=1 (bath_up_L), bathIdx=3 (bath_dn_L) pour le courant total

4. numerical_derivative(Z, X) calculait dX/dZ (résistance) au lieu de dZ/dX
   → corrigé : G = dI/dΦ avec différences centrées

5. Grille phi=-2:0.05:2 trop étroite (structures à |Φ|≈10 invisibles)
   → phi = range(-15, 15, length=61)

6. Figures avec même mise en page publication que DOS et populations :
   2 panneaux (fort/faible couplage) + légende externe
   Mêmes palettes de couleurs hex que les figures précédentes

Référence API : https://qutip.org/qutip-julia-tutorials/HierarchicalEOM.jl/electronic_current.html
═══════════════════════════════════════════════════════════════════
"""

using HierarchicalEOM
using CairoMakie, Colors
using Printf

CairoMakie.activate!(type = "png")

# ── Couleurs — même convention que DOS et populations ──────────────────────
function hex2rgb(h::String)
    r = parse(Int, h[1:2], base=16) / 255f0
    g = parse(Int, h[3:4], base=16) / 255f0
    b = parse(Int, h[5:6], base=16) / 255f0
    return RGBf(r, g, b)
end

# Fort couplage : bleu foncé → clair (Γ = 1.0 → 2.0)
KONDO_COLORS = [hex2rgb("08306B"), hex2rgb("2171B5"), hex2rgb("9ECAE1")]
KONDO_LW     = [2.2, 1.7, 1.2]
KONDO_LS     = [:solid, :dashdot, :dash]

# Frontière
BOUNDARY_COLOR = RGBf(0.1f0, 0.1f0, 0.1f0)

# Faible couplage : rouge-orange clair → foncé (Γ = 7.0 → 15.0)
METAL_COLORS = [hex2rgb("F4A582"), hex2rgb("D6604D"), hex2rgb("B2182B")]
METAL_LW     = [1.2, 1.7, 2.2]
METAL_LS     = [:dash, :dashdot, :solid]

BG_KONDO = RGBAf(0.94f0, 0.96f0, 0.98f0, 1f0)   # bleu pâle
BG_METAL = RGBAf(0.98f0, 0.95f0, 0.93f0, 1f0)   # orange pâle

# ── Paramètres physiques ───────────────────────────────────────────────────
const ε  = -5.0
const U  = 10.0
const kT = 0.5
const W  = 10.0
const N  = 5      # termes Padé
const tier = 3    # profondeur hiérarchie

# Mêmes 7 valeurs de Γ que tous les autres scripts
Gamma_list = [1.0, 1.5, 2.0, 3.18, 7.0, 10.0, 15.0]

# Grille de biais étendue pour voir les marches de Hubbard à |Φ|≈10
phi_list = range(-15.0, 15.0, length=61)

# ── Hamiltonien SIAM (identique aux autres scripts) ────────────────────────
σm = sigmam()
σz = sigmaz()
II = qeye(2)
d_up = tensor(σm, II)
d_dn = tensor(-1 * σz, σm)
Hsys = ε * (d_up' * d_up + d_dn' * d_dn) + U * (d_up' * d_up * d_dn' * d_dn)

# État initial : impureté vide
# rho_0 non utilisé : steadystate ne nécessite pas d'état initial

# ── Fonction de courant (API officielle HierarchicalEOM.jl) ───────────────
# Référence : https://qutip.org/qutip-julia-tutorials/HierarchicalEOM.jl/
#             electronic_current.html
# bathIdx : 1 = bath_up_L, 2 = bath_up_R, 3 = bath_dn_L, 4 = bath_dn_R
function Ic(ados, M::M_Fermion, bathIdx::Int)
    HDict = M.hierarchy
    idx_list = HDict.lvl2idx[1]
    I = 0.0im
    for idx in idx_list
        ρ1 = ados[idx]
        nvec = HDict.idx2nvec[idx]
        for (α, k, _) in getIndexEnsemble(nvec, HDict.bathPtr)
            if α == bathIdx
                exponent = M.bath[α][k]
                if exponent.types == "fA"
                    I += tr(exponent.op' * ρ1)
                elseif exponent.types == "fE"
                    I -= tr(exponent.op' * ρ1)
                end
                break
            end
        end
    end
    # Conversion en unités adimensionnées (ħ = e = 1)
    # Courant positif = flux L → système → R
    return real(1im * I)
end

# ── Dérivée numérique centrée dI/dΦ ───────────────────────────────────────
function dIdPhi(phi_arr, I_arr)
    n = length(I_arr)
    G = zeros(n)
    G[1]   = (I_arr[2] - I_arr[1])   / (phi_arr[2] - phi_arr[1])
    G[end] = (I_arr[end] - I_arr[end-1]) / (phi_arr[end] - phi_arr[end-1])
    for i in 2:n-1
        G[i] = (I_arr[i+1] - I_arr[i-1]) / (phi_arr[i+1] - phi_arr[i-1])
    end
    return G
end

# ══════════════════════════════════════════════════════════════════════════
# BOUCLE PRINCIPALE : steadystate à chaque (Γ, Φ)
# API correcte : reconstruire les 4 bains à chaque biais,
#                reconstruire M_even, puis appeler steadystate(M_even)
# ══════════════════════════════════════════════════════════════════════════
phi_arr = collect(phi_list)
results = Dict{Float64, NamedTuple}()

for Gamma in Gamma_list
    ratio = U / (π * Gamma)
    println("\n" * "="^60)
    @printf("  Γ = %.2f   U/πΓ = %.3f   (%d points de biais)\n",
            Gamma, ratio, length(phi_arr))
    println("="^60)

    I_arr = zeros(length(phi_arr))

    for (i, Phi) in enumerate(phi_arr)
        mu_L =  Phi / 2.0
        mu_R = -Phi / 2.0

        # 4 bains : spin-up et spin-down pour L et R
        bath_up_L = Fermion_Lorentz_Pade(d_up, Gamma, mu_L, W, kT, N)
        bath_up_R = Fermion_Lorentz_Pade(d_up, Gamma, mu_R, W, kT, N)
        bath_dn_L = Fermion_Lorentz_Pade(d_dn, Gamma, mu_L, W, kT, N)
        bath_dn_R = Fermion_Lorentz_Pade(d_dn, Gamma, mu_R, W, kT, N)

        M_even = M_Fermion(Hsys, tier, [bath_up_L, bath_up_R,
                                        bath_dn_L, bath_dn_R])

        # État stationnaire exact par résolution linéaire
        ados_ss = steadystate(M_even)

        # Courant total depuis le bain gauche (↑ + ↓)
        I_upL = Ic(ados_ss, M_even, 1)
        I_dnL = Ic(ados_ss, M_even, 3)
        I_arr[i] = I_upL + I_dnL

        if i % 10 == 0 || i == 1
            @printf("    Φ = %+6.2f  I = %+.4f\n", Phi, I_arr[i])
        end
    end

    G_arr = dIdPhi(phi_arr, I_arr)
    results[Gamma] = (I=I_arr, G=G_arr, ratio=ratio)

    @printf("  → I_max = %.4f   G(0) = %.4f\n",
            maximum(abs.(I_arr)), G_arr[length(phi_arr)÷2 + 1])
end

println("\n✓ Calcul terminé. Génération des figures...\n")

# ══════════════════════════════════════════════════════════════════════════
# HELPERS FIGURE — style identique aux figures DOS et populations
# ══════════════════════════════════════════════════════════════════════════
function get_line_style(Gamma)
    ratio = U / (π * Gamma)
    if abs(ratio - 1.0) < 0.02
        return BOUNDARY_COLOR, 3.0, :solid, ratio
    elseif ratio > 1.0
        idx = argmin(abs.([1.0, 1.5, 2.0] .- Gamma))
        return KONDO_COLORS[idx], KONDO_LW[idx], KONDO_LS[idx], ratio
    else
        idx = argmin(abs.([7.0, 10.0, 15.0] .- Gamma))
        return METAL_COLORS[idx], METAL_LW[idx], METAL_LS[idx], ratio
    end
end

# ── Handles de légende ─────────────────────────────────────────────────────
function make_legend()
    legend_entries = []
    legend_labels  = String[]

    # Fort couplage
    for (Gamma, col, lw, ls) in zip([1.0, 1.5, 2.0],
                                     KONDO_COLORS, KONDO_LW, KONDO_LS)
        ratio = U / (π * Gamma)
        push!(legend_entries, LineElement(color=col, linewidth=lw, linestyle=ls))
        push!(legend_labels, @sprintf("U/πΓ = %.2f  (Γ=%.1f)", ratio, Gamma))
    end
    # Frontière
    ratio_b = U / (π * 3.18)
    push!(legend_entries, LineElement(color=BOUNDARY_COLOR, linewidth=3.0, linestyle=:solid))
    push!(legend_labels, @sprintf("U/πΓ = %.2f  (Γ=3.18, boundary)", ratio_b))
    # Faible couplage
    for (Gamma, col, lw, ls) in zip([7.0, 10.0, 15.0],
                                     METAL_COLORS, METAL_LW, METAL_LS)
        ratio = U / (π * Gamma)
        push!(legend_entries, LineElement(color=col, linewidth=lw, linestyle=ls))
        push!(legend_labels, @sprintf("U/πΓ = %.2f  (Γ=%.1f)", ratio, Gamma))
    end
    return legend_entries, legend_labels
end

# ══════════════════════════════════════════════════════════════════════════
# FIGURE 1 — Courant I(Φ)
# ══════════════════════════════════════════════════════════════════════════
fig1 = Figure(size=(1050, 420),
              figure_padding=(12, 12, 10, 10))

ax1a = Axis(fig1[1, 1];
    backgroundcolor = BG_KONDO,
    xlabel          = L"\Phi \; [\Gamma^{-1}]",
    ylabel          = L"I(\Phi) \; [\Gamma]",
    title           = "Strong coupling  (U/πΓ ≥ 1)",
    titlecolor      = hex2rgb("1a4a8a"),
    titlesize       = 13f0,
    xminorticks     = IntervalsBetween(4),
    yminorticks     = IntervalsBetween(4),
    xminorticksvisible = true,
    yminorticksvisible = true,
)

ax1b = Axis(fig1[1, 2];
    backgroundcolor = BG_METAL,
    xlabel          = L"\Phi \; [\Gamma^{-1}]",
    title           = "Weak coupling  (U/πΓ ≤ 1)",
    titlecolor      = hex2rgb("8B1A1A"),
    titlesize       = 13f0,
    yticklabelsvisible = false,
    xminorticks     = IntervalsBetween(4),
    yminorticks     = IntervalsBetween(4),
    xminorticksvisible = true,
    yminorticksvisible = true,
)

linkyaxes!(ax1a, ax1b)

# Tracé fort couplage + frontière
for Gamma in [1.0, 1.5, 2.0, 3.18]
    col, lw, ls, ratio = get_line_style(Gamma)
    lines!(ax1a, phi_arr, results[Gamma].I; color=col, linewidth=lw, linestyle=ls)
end

# Tracé faible couplage + frontière
for Gamma in [3.18, 7.0, 10.0, 15.0]
    col, lw, ls, ratio = get_line_style(Gamma)
    lines!(ax1b, phi_arr, results[Gamma].I; color=col, linewidth=lw, linestyle=ls)
end

# Lignes de référence
for ax in [ax1a, ax1b]
    hlines!(ax, [0.0]; color=:black, linewidth=0.8)
    vlines!(ax, [0.0]; color=(:black, 0.5), linewidth=0.6, linestyle=:dot)
end

# Annotations de panneau
text!(ax1a, "(a)"; position=Point2f(-14.0, maximum(results[1.0].I)*0.92),
      fontsize=13, font="TeX Gyre Heros Bold")
text!(ax1b, "(b)"; position=Point2f(-14.0, maximum(results[1.0].I)*0.92),
      fontsize=13, font="TeX Gyre Heros Bold")

# Légende
leg_entries, leg_labels = make_legend()
Legend(fig1[1, 3], leg_entries, leg_labels;
    framevisible    = true,
    labelsize       = 9f0,
    titlesize       = 9.5f0,
    title           = "U/πΓ  (darkness ∝ coupling)",
    titlegap        = 4f0,
    rowgap          = 4f0,
    padding         = (8f0, 8f0, 6f0, 6f0),
    patchsize       = (28f0, 12f0),
)

Label(fig1[0, 1:2];
    text      = @sprintf("Steady-state current  I(Φ) — SIAM  (ε=%d, U=%d, kT=%.1f, ħ=1,  tier=%d, N=%d)",
                         Int(ε), Int(U), kT, tier, N),
    fontsize  = 13f0,
    font      = "TeX Gyre Heros Bold",
    padding   = (0f0, 0f0, 4f0, 0f0),
)

colsize!(fig1.layout, 1, Relative(0.42))
colsize!(fig1.layout, 2, Relative(0.42))
colsize!(fig1.layout, 3, Relative(0.16))

save("IV_current_jl.pdf", fig1)
save("IV_current_jl.png", fig1, px_per_unit=2)
println("✓ Figure 1 sauvegardée : IV_current_jl.pdf / .png")

# ══════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Conductance G(Φ) = dI/dΦ
# ══════════════════════════════════════════════════════════════════════════
fig2 = Figure(size=(1050, 420),
              figure_padding=(12, 12, 10, 10))

ax2a = Axis(fig2[1, 1];
    backgroundcolor = BG_KONDO,
    xlabel          = L"\Phi \; [\Gamma^{-1}]",
    ylabel          = L"G(\Phi) = dI/d\Phi \; [\Gamma^{-1}]",
    title           = "Strong coupling  (U/πΓ ≥ 1)",
    titlecolor      = hex2rgb("1a4a8a"),
    titlesize       = 13f0,
    xminorticks     = IntervalsBetween(4),
    yminorticks     = IntervalsBetween(4),
    xminorticksvisible = true,
    yminorticksvisible = true,
)

ax2b = Axis(fig2[1, 2];
    backgroundcolor = BG_METAL,
    xlabel          = L"\Phi \; [\Gamma^{-1}]",
    title           = "Weak coupling  (U/πΓ ≤ 1)",
    titlecolor      = hex2rgb("8B1A1A"),
    titlesize       = 13f0,
    yticklabelsvisible = false,
    xminorticks     = IntervalsBetween(4),
    yminorticks     = IntervalsBetween(4),
    xminorticksvisible = true,
    yminorticksvisible = true,
)

linkyaxes!(ax2a, ax2b)

# Tracé conductance fort couplage
for Gamma in [1.0, 1.5, 2.0, 3.18]
    col, lw, ls, ratio = get_line_style(Gamma)
    lines!(ax2a, phi_arr, results[Gamma].G; color=col, linewidth=lw, linestyle=ls)
end

# Tracé conductance faible couplage
for Gamma in [3.18, 7.0, 10.0, 15.0]
    col, lw, ls, ratio = get_line_style(Gamma)
    lines!(ax2b, phi_arr, results[Gamma].G; color=col, linewidth=lw, linestyle=ls)
end

# Lignes de référence et marches de Hubbard
for ax in [ax2a, ax2b]
    hlines!(ax, [0.0]; color=:black, linewidth=0.8)
    vlines!(ax, [0.0]; color=(:black, 0.5), linewidth=0.6, linestyle=:dot)
    # Marches de Hubbard : Φ = ±2|ε| = ±10 et Φ = ±2|ε+U| = 0
    for xref in [2*abs(ε), 2*abs(ε + U)]
        vlines!(ax, [ xref];  color=(:crimson, 0.5),     linewidth=0.9, linestyle=:dash)
        vlines!(ax, [-xref];  color=(:crimson, 0.5),     linewidth=0.9, linestyle=:dash)
    end
end

# Annotations de panneau
text!(ax2a, "(a)"; position=Point2f(-14.0, 0.0), fontsize=13, font="TeX Gyre Heros Bold")
text!(ax2b, "(b)"; position=Point2f(-14.0, 0.0), fontsize=13, font="TeX Gyre Heros Bold")

# Annotation des marches (panneau a)
text!(ax2a, "2|ε|";
      position=Point2f(2*abs(ε) + 0.3, 0.0),
      fontsize=9, color=:crimson, align=(:left, :bottom))

# Légende enrichie (+ marches de Hubbard)
leg_entries2, leg_labels2 = make_legend()
push!(leg_entries2, LineElement(color=(:crimson, 0.6), linewidth=0.9, linestyle=:dash))
push!(leg_labels2,  @sprintf("2|ε| = %.1f  (Hubbard steps)", 2*abs(ε)))

Legend(fig2[1, 3], leg_entries2, leg_labels2;
    framevisible    = true,
    labelsize       = 9f0,
    titlesize       = 9.5f0,
    title           = "U/πΓ  (darkness ∝ coupling)",
    titlegap        = 4f0,
    rowgap          = 4f0,
    padding         = (8f0, 8f0, 6f0, 6f0),
    patchsize       = (28f0, 12f0),
)

Label(fig2[0, 1:2];
    text      = @sprintf("Differential conductance  G(Φ) = dI/dΦ — SIAM  (ε=%d, U=%d, kT=%.1f, ħ=1,  tier=%d, N=%d)",
                         Int(ε), Int(U), kT, tier, N),
    fontsize  = 13f0,
    font      = "TeX Gyre Heros Bold",
    padding   = (0f0, 0f0, 4f0, 0f0),
)

colsize!(fig2.layout, 1, Relative(0.42))
colsize!(fig2.layout, 2, Relative(0.42))
colsize!(fig2.layout, 3, Relative(0.16))

save("IV_conductance_jl.pdf", fig2)
save("IV_conductance_jl.png", fig2, px_per_unit=2)
println("✓ Figure 2 sauvegardée : IV_conductance_jl.pdf / .png")

# ── Résumé numérique ──────────────────────────────────────────────────────
println("\n" * "="^65)
println("RÉSUMÉ PHYSIQUE")
println("="^65)
@printf("  Γ* = U/π = %.3f\n", U/π)
@printf("  Marches Hubbard : Φ ≈ ±%.1f (LHB), ±%.1f (UHB)\n\n",
        2*abs(ε), 2*abs(ε+U))

for Gamma in Gamma_list
    r = results[Gamma]
    regime = r.ratio > 1.02 ? "Kondo " :
             abs(r.ratio - 1.0) < 0.02 ? "border" : "metal "
    mid = length(phi_arr)÷2 + 1
    @printf("  Γ=%5.2f  U/πΓ=%.3f  [%s]  I_max=%+.4f  G(0)=%.4f\n",
            Gamma, r.ratio, regime, maximum(abs.(r.I)), r.G[mid])
end


  Γ = 1.00   U/πΓ = 3.183   (61 points de biais)
Preparing block matrices for HEOM Liouvillian superoperator (using 1 threads)...
Progress: [==============================] 100.0% --- Elapsed Time: 0h 00m 06s (ETA: 0h 00m 00s))
Constructing matrix...[DONE]
Solving steady state for ADOs by linear-solve method...
Calculating left preconditioner with ilu...[DONE]
Solving linear problem...[DONE]
    Φ = -15.00  I = -0.4902
Preparing block matrices for HEOM Liouvillian superoperator (using 1 threads)...
Progress: [==============================] 100.0% --- Elapsed Time: 0h 01m 03s (ETA: 0h 00m 00s)
Constructing matrix...[DONE]
Solving steady state for ADOs by linear-solve method...
Calculating left preconditioner with ilu...